# Session 06: Pydantic and Data Validation

**Notebook 1 of 2: Concepts** · used during the live session and for revision afterwards (about 60 minutes on your own)

Since Session 03 you've written validation by hand: `validate_record` for tickets, `load_config` for configuration, `Asset.from_json` and `load_settings` for the inventory. Each one re-implemented the same things: required fields, type checks, defaults, error messages. **Pydantic** replaces all of that with declarations: you describe the data once, as a class with type hints, and Pydantic validates, converts, documents and serialises it.

By the end of this notebook you will be able to:

1. **Define** Pydantic models with typed, optional, defaulted, constrained and nested fields.
2. **Interpret** a `ValidationError`, and **map** it to a user-facing message.
3. **Write** `field_validator` and `model_validator` rules for business constraints.
4. **Serialise** with `model_dump` and `model_dump_json`, controlling aliases and exclusions.
5. **Load** typed configuration from the environment with `pydantic-settings`, including secrets.

> **Kernel:** the course environment, after `uv sync`. Pydantic and pydantic-settings are in its `web` group. Everything here runs offline.

## The engineering problem

The Orbital Logistics (fictional) incident tooling has grown three hand-written validators: about 50 lines for ticket records, 35 for `config.json` and 25 for the inventory settings. A product owner now asks for one more ticket field, `"region"`, which is optional, must be one of four values, and defaults to `"eu"`. Adding it by hand means touching the type check, the default, the error message and the tests, in a function that's already hard to read.

Next session, the same ticket rules must also validate **HTTP requests** to a FastAPI service, and appear in its **API documentation**. A third copy of the rules would be a maintenance problem.

Pydantic puts the rules in one place, the model, and everything else (validation, conversion, error messages, JSON, settings, and in Session 07 the API docs) is derived from it.

## Part 1: A first model

A Pydantic model is a class that inherits from `BaseModel`, with fields declared by type hints, exactly like a dataclass. The difference is what happens when you create one: **the input is validated and converted**.

In [ ]:
from pydantic import BaseModel, ValidationError


class Asset(BaseModel):
    id: str
    name: str
    team: str
    criticality: str
    open_tickets: int = 0


raw = {
    "id": "db-03",
    "name": "Orders database replica",
    "team": "database",
    "criticality": "high",
    "open_tickets": "2",
}

asset = Asset.model_validate(raw)  # or Asset(**raw)
print(asset)
print(asset.open_tickets + 1, type(asset.open_tickets).__name__)

**Explain:** `"2"` arrived as a string and became the integer `2`, because the field is declared `int`. In Session 03 you'd have written that check by hand; here it comes from the annotation. Unlike a dataclass, the hint is **enforced at run time**, which is the gap Session 03 Part 4 promised Pydantic would fill.

**Predict:** what happens with three things wrong at once?

In [ ]:
broken = {"id": "db-03", "team": 7, "criticality": "high", "open_tickets": "several"}

try:
    Asset.model_validate(broken)
except ValidationError as error:
    print(error)

**Explain:** one `ValidationError` lists **every** problem: `name` is missing, `team` isn't a string, and `open_tickets` can't be parsed as an integer. That's the "collect all problems" pattern you wrote by hand in Session 03, now automatic. Notice that `team=7` was rejected rather than converted to `"7"`: Pydantic converts only where it's unambiguous.

![How Pydantic validates input](../../../assets/diagrams/pydantic-validation-flow.svg)

**How this works**

- Validation runs in stages. First each field's value is **converted** to its declared type. Then declared **constraints** are checked (minimum values, patterns, allowed literals). Then your own **field validators** run, then **model validators**, which can see all fields at once.
- A failure in one field doesn't stop the others. Every problem from every stage is collected into **one** `ValidationError`, so one run tells the data owner everything that's wrong.
- `error.errors()` gives each problem as a dict: `loc` (where, as a path such as `("tags", 1)`), `type` (a stable code such as `int_type`, good for programs) and `msg` (a sentence for people).
- Conversion is **lax** by default: `"80"` becomes `80`, and so does `True` → `1`. That's convenient for forms and environment variables, where everything is text, and wrong for a data contract where `"80"` signals a bug upstream. **Strict** mode, per field or per model, refuses conversions.
- If all stages pass, you hold a typed instance you can trust: code after this point needs no defensive checks.

| | Java | C# | TypeScript | Python |
|---|---|---|---|---|
| Declare rules on fields | Bean Validation: `@NotNull`, `@Min(0)` | DataAnnotations: `[Required]`, `[Range]` | zod: `z.number().min(0)` | type hints plus `Field(ge=0)` |
| Validate | `validator.validate(obj)` | `Validator.TryValidateObject(...)` | `schema.parse(data)` | `Model.model_validate(data)` |
| Errors | `Set<ConstraintViolation>` | `List<ValidationResult>` | `ZodError.issues` | `ValidationError.errors()` |
| Parse JSON too | Jackson, separately | System.Text.Json, separately | `JSON.parse`, then zod | `Model.model_validate_json(text)`, in one step |

## Part 2: Field types and constraints

### Required, optional, defaulted

Three different things that are easy to confuse:

In [ ]:
class TicketDraft(BaseModel):
    id: str  # required, must be a string
    assignee: str | None  # REQUIRED, but may be null: the key must be present
    note: str | None = None  # optional: may be missing or null, defaults to None
    priority: str = "P3"  # optional with a default


print(TicketDraft.model_validate({"id": "INC-1", "assignee": None}))

try:
    TicketDraft.model_validate({"id": "INC-1"})
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])

**Explain:** `str | None` alone means "**must be provided**, and may be `None`". Only a **default** makes a field optional. So `assignee` is required even though `None` is allowed, which is exactly the rule the incident exports need: the field must exist, and `null` means unassigned.

For mutable defaults, use `Field(default_factory=list)`, as with dataclasses. (Pydantic copies a plain `[]` default for each instance, so it's actually safe here, but `default_factory` states the intent and works the same in both.)

### Constraints and allowed values

`Field(...)` declares constraints; `Literal` restricts a field to fixed values:

In [ ]:
from typing import Literal

from pydantic import Field


class Ticket(BaseModel):
    id: str = Field(pattern=r"^INC-\d+$")
    title: str = Field(min_length=1, max_length=200)
    priority: Literal["P1", "P2", "P3", "P4"]
    age_hours: int = Field(ge=0)
    region: Literal["eu", "us", "apac", "latam"] = "eu"
    tags: list[str] = Field(default_factory=list)


good = Ticket.model_validate(
    {"id": "INC-3004", "title": "Disk full", "priority": "P3", "age_hours": 80}
)
print(good)

try:
    Ticket.model_validate(
        {"id": "3004", "title": "", "priority": "P9", "age_hours": -1, "tags": ["dns", 4]}
    )
except ValidationError as error:
    for issue in error.errors():
        print(f"{'.'.join(map(str, issue['loc'])):<10} {issue['type']:<22} {issue['msg']}")

That's the product owner's `region` request from the engineering problem: **one line**. Five rules were broken, and five errors came back, each with its location. `tags.1` points at the second element of the list.

### Lax versus strict

**Predict** which of these inputs each model accepts for `age_hours`:

In [ ]:
class LaxTicket(BaseModel):
    age_hours: int


class StrictTicket(BaseModel):
    age_hours: int = Field(strict=True)


for value in [80, "80", True, 80.0, 80.5]:
    results = []
    for model in (LaxTicket, StrictTicket):
        try:
            results.append(repr(model.model_validate({"age_hours": value}).age_hours))
        except ValidationError as error:
            results.append("✗ " + error.errors()[0]["type"])
    print(f"{value!r:<6} lax: {results[0]:<20} strict: {results[1]}")

**Explain:** lax mode converts anything unambiguous, including `True` → `1` (the `bool`-is-an-`int` trap from Sessions 02 and 03) and `80.0` → `80`, but refuses `80.5`, which would lose data. Strict mode accepts only a real `int`.

**Engineering Insight:** decide per boundary. Configuration from environment variables *must* be lax, because everything arrives as text. An export from another system is a contract: a string where a number belongs is a bug in that system, and silently converting it hides the bug. The workbook makes `age_hours` strict for exactly this reason. You can set strictness per field (`Field(strict=True)`) or for the whole model (`model_config = ConfigDict(strict=True)`); model-wide strict mode also refuses a JSON list for a `tuple` field when validating Python objects, so per-field strictness is often the more practical choice.

### Nested models and unknown fields

Models can contain models, lists of models and dicts of models. Validation goes all the way down, and error locations show the path. By default, **unknown fields are ignored**; `extra="forbid"` rejects them instead:

In [ ]:
from pydantic import ConfigDict


class Owner(BaseModel):
    team: str
    on_call: str


class InventoryAsset(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: str
    owner: Owner
    related: list[str] = Field(default_factory=list)


data = {"id": "db-03", "owner": {"team": "database", "on_call": 7}, "colour": "blue"}
try:
    InventoryAsset.model_validate(data)
except ValidationError as error:
    for issue in error.errors():
        print(".".join(map(str, issue["loc"])), "->", issue["msg"])

| `extra=` | Unknown field | Use when |
|---|---|---|
| `"ignore"` (default) | Silently dropped | Reading other systems' data, which gain fields over time |
| `"forbid"` | Validation error | Your own requests and config, where an unknown key is probably a typo |
| `"allow"` | Kept on the instance | Rarely; pass-through of data you don't model |

## Part 3: Turning errors into messages

`ValidationError`'s default text is written for developers. Users and data owners need something shorter. Because `errors()` is structured, the mapping is a few lines. The workbook's `Rejection.from_validation_error` does exactly this:

In [ ]:
def describe(error: ValidationError) -> list[str]:
    """One 'field: message' line per problem, for reports and logs."""
    return [
        f"{'.'.join(str(part) for part in issue['loc']) or 'record'}: {issue['msg']}"
        for issue in error.errors(include_url=False)
    ]


try:
    Ticket.model_validate({"id": "INC-1", "title": "x", "priority": "P1", "age_hours": "soon"})
except ValidationError as error:
    print(describe(error))

try:
    Ticket.model_validate(["not", "a", "ticket"])
except ValidationError as error:
    print(describe(error))

The second case has an empty `loc`: the problem is the whole input, not a field, so the helper labels it `record`.

**Engineering Insight:** branch on `issue["type"]` (`missing`, `int_type`, `literal_error` ...) if code needs to react differently to different problems. The `msg` text can change between Pydantic versions; the type codes are part of its documented API.

### Validating JSON text directly

`model_validate_json` parses and validates in one step. It's faster than `json.loads` followed by `model_validate`, and in **strict** mode it understands JSON's types (a JSON array is acceptable for a `tuple` field, for example):

In [ ]:
text = (
    '{"id": "INC-3004", "title": "Disk full", "priority": "P3",'
    ' "age_hours": 80, "tags": ["storage"]}'
)
print(Ticket.model_validate_json(text))

try:
    Ticket.model_validate_json('{"id": "INC-3004", "title": "Disk full",')
except ValidationError as error:
    print(error.errors()[0]["type"], "->", error.errors()[0]["msg"])

Malformed JSON becomes a `json_invalid` validation error rather than a `JSONDecodeError`. The workbook still reads export files with `json.load`, because a whole-file problem (not JSON, or not a list) must be reported differently from a bad record, as in Session 03.

## Part 4: Your own rules

Types and constraints cover most needs. For the rest, write validators.

### Field validators

A `@field_validator` runs for one or more fields. In the default **after** mode it receives the already-converted value and returns the value to store, so it's the place to **normalise**. In **before** mode it receives the raw input, so it can reshape data before conversion. To reject a value, raise `ValueError`, and Pydantic turns it into a normal validation error.

In [ ]:
from pydantic import field_validator


class CleanTicket(BaseModel):
    id: str
    team: str
    assignee: str | None
    tags: tuple[str, ...]

    @field_validator("team")
    @classmethod
    def lower_team(cls, value: str) -> str:
        return value.strip().lower()

    @field_validator("assignee")
    @classmethod
    def blank_is_none(cls, value: str | None) -> str | None:
        return (value or "").strip() or None

    @field_validator("tags", mode="before")
    @classmethod
    def split_comma_separated(cls, value: object) -> object:
        if isinstance(value, str):  # a legacy system sends "dns,Network"
            return [part.strip() for part in value.split(",") if part.strip()]
        return value

    @field_validator("tags")
    @classmethod
    def normalise_tags(cls, value: tuple[str, ...]) -> tuple[str, ...]:
        if len(value) > 10:
            raise ValueError("at most 10 tags")
        return tuple(sorted({tag.lower() for tag in value}))


print(
    CleanTicket.model_validate(
        {"id": "INC-1", "team": " Platform ", "assignee": "  ", "tags": "dns,Network,dns"}
    )
)
print(
    CleanTicket.model_validate(
        {"id": "INC-2", "team": "network", "assignee": "li", "tags": ["VPN"]}
    )
)

try:
    CleanTicket.model_validate(
        {"id": "INC-3", "team": "x", "assignee": None, "tags": [f"t{n}" for n in range(11)]}
    )
except ValidationError as error:
    print(describe(error))

That's Session 02's `normalise_ticket` and part of Session 03's `validate_record`, declared next to the fields they apply to. The model now **guarantees** that every `CleanTicket` is normalised: there's no way to create one that isn't.

**Common Mistake:** forgetting `return value` in a validator. The field silently becomes `None`. And raise only `ValueError` (or `AssertionError`) inside validators; other exceptions aren't converted into validation errors, so they escape as crashes.

### Model validators

A rule involving **several fields** needs a `@model_validator`. In `"after"` mode it runs on the finished instance and returns it:

In [ ]:
from pydantic import model_validator


class ResolvedTicket(BaseModel):
    id: str
    status: Literal["open", "in_progress", "resolved", "closed"]
    resolution: str | None = None

    @model_validator(mode="after")
    def resolved_tickets_need_a_resolution(self) -> "ResolvedTicket":
        if self.status in ("resolved", "closed") and not self.resolution:
            raise ValueError(f"{self.status} tickets need a resolution")
        return self


print(ResolvedTicket.model_validate({"id": "INC-1", "status": "open"}))
try:
    ResolvedTicket.model_validate({"id": "INC-2", "status": "resolved"})
except ValidationError as error:
    print(describe(error))

The error's location is empty, because the rule concerns the whole record (our helper labels it `record`). The workbook's `Settings` model uses a model validator for "SLA hours must increase from P1 to P4", a rule no single field can express.

**Modify:** add a rule to `ResolvedTicket` that `resolution` must be empty while the ticket is `open`. Which kind of validator is it?

## Part 5: Serialisation

Models turn back into data with `model_dump` (a dict) and `model_dump_json` (a string). Nested models are converted at every level.

![A nested model and the JSON it becomes](../../../assets/diagrams/nested-model-structure.svg)

**How this works**

- A model can contain other models, lists of models and dicts of models. The workbook's `Report` holds `Totals`, a dict of `TeamStats`, and lists of `Breach`, `TagCount` and `Rejection`.
- `model_dump_json()` walks the whole tree: models become JSON objects, lists and tuples become arrays, `None` becomes `null`, and a `datetime` becomes an ISO 8601 string (`Z` for UTC).
- `model_validate_json()` does the reverse, validating every level on the way in. A saved report can be loaded back as a trustworthy object, and a test can check a round trip with a single `==`.
- `include`, `exclude` and `exclude_none` shape the output without changing the model. The workbook's `--summary-only` option is one `exclude`.
- Aliases decouple Python names from external names, which is essential when an API uses `camelCase` and your code uses `snake_case`.

In [ ]:
from datetime import UTC, datetime


class Breach(BaseModel):
    id: str
    team: str
    age_hours: int


class Summary(BaseModel):
    generated_at: datetime
    breaches: list[Breach]
    note: str | None = None


summary = Summary(
    generated_at=datetime(2026, 10, 21, 2, 0, tzinfo=UTC),
    breaches=[Breach(id="INC-3007", team="it-support", age_hours=200)],
)

print(summary.model_dump())  # Python objects: datetime stays a datetime
print(summary.model_dump(mode="json"))  # JSON-compatible: datetime becomes a string
print(summary.model_dump_json(exclude_none=True))  # a JSON string, without null fields
print(summary.model_dump_json(include={"breaches": {0: {"id"}}}))
print(Summary.model_validate_json(summary.model_dump_json()) == summary)

### Aliases: different names outside

Suppose version 2 of the Asset Inventory API sends `camelCase` keys. Declare an **alias** for the external name and keep Python names Pythonic:

In [ ]:
from pydantic.alias_generators import to_camel


class AssetV2(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel, populate_by_name=True)

    asset_id: str
    owning_team: str
    open_ticket_count: int = 0


incoming = {"assetId": "db-03", "owningTeam": "database", "openTicketCount": 2}
asset = AssetV2.model_validate(incoming)

print(asset.owning_team)  # Python name
print(asset.model_dump())  # Python names
print(asset.model_dump(by_alias=True))  # external names, for sending back
print(AssetV2(asset_id="vpn-ber", owning_team="network"))  # populate_by_name: Python names work too

For a single field, `Field(alias="ticketId")` does the same thing. If the input and output names differ, use `validation_alias` and `serialization_alias` separately. FastAPI (Session 07) uses `by_alias=True` when it writes responses, so the aliases become your API's public field names.

## Part 6: Settings with `pydantic-settings`

`BaseSettings` is a model whose values come from **environment variables** and an optional **`.env` file**, instead of from a dict. Everything from this notebook applies: types, defaults, constraints, validators. Environment variables are text, so conversion is lax by design: `"30"` becomes `30`, `"true"` becomes `True`, and a JSON string becomes a dict or list.

In [ ]:
import os
import tempfile
from pathlib import Path

from pydantic import PositiveInt, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class InventorySettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="INVENTORY_", env_file=".env", extra="ignore")

    base_url: str = "http://127.0.0.1:8765"
    api_token: SecretStr
    timeout_seconds: PositiveInt = 5
    verify_tls: bool = True
    team_channels: dict[str, str] = Field(default_factory=dict)


with tempfile.TemporaryDirectory() as workdir:
    original_dir = Path.cwd()
    os.chdir(workdir)
    try:
        Path(".env").write_text(
            "INVENTORY_API_TOKEN=not-a-real-token\nINVENTORY_TIMEOUT_SECONDS=30\n", encoding="utf-8"
        )
        os.environ["INVENTORY_VERIFY_TLS"] = "false"
        os.environ["INVENTORY_TEAM_CHANNELS"] = '{"database": "#db-oncall"}'

        settings = InventorySettings()
        print(settings)
        print(settings.timeout_seconds + 1, settings.verify_tls, settings.team_channels["database"])
        print(
            "token for the Authorization header:", settings.api_token.get_secret_value()[:4] + "..."
        )
    finally:
        os.chdir(original_dir)
        for name in ["INVENTORY_VERIFY_TLS", "INVENTORY_TEAM_CHANNELS"]:
            os.environ.pop(name, None)

**Explain:**

- `env_prefix="INVENTORY_"` maps the field `api_token` to the variable `INVENTORY_API_TOKEN`. Matching is case-insensitive by default.
- Values came from both places: the token and timeout from `.env`, `verify_tls` and `team_channels` from the environment. When both define a variable, **the environment wins**, the same precedence you implemented by hand in Session 05.
- `"30"` became an `int`, `"false"` a `bool`, and the JSON text a `dict`.
- **`SecretStr`** shows `'**********'` in `repr()`, in logs and in tracebacks. You get the real value only by asking explicitly with `.get_secret_value()`. That's Session 05's `field(repr=False)`, built in, and it also protects against `print(settings.api_token)`.

Two kinds of failure, worth knowing apart:

In [ ]:
from pydantic_settings import SettingsError

for name, value in [
    ("INVENTORY_TIMEOUT_SECONDS", "0"),
    ("INVENTORY_TEAM_CHANNELS", "database=#db"),
]:
    os.environ["INVENTORY_API_TOKEN"] = "not-a-real-token"
    os.environ[name] = value
    try:
        InventorySettings(_env_file=None)
    except ValidationError as error:
        print(f"{name}={value!r}: ValidationError ->", describe(error))
    except SettingsError as error:
        print(f"{name}={value!r}: SettingsError ->", error)
    finally:
        os.environ.pop(name, None)
        os.environ.pop("INVENTORY_API_TOKEN", None)

**Explain:** a value that parses but breaks a rule (`0` isn't a `PositiveInt`) is a normal `ValidationError`. A value that can't even be **parsed** (a dict field whose variable isn't JSON) raises `pydantic_settings.SettingsError` before validation starts. A command-line tool should catch **both** and print a clear message; the workbook's provided `cli.py` does exactly that, after a learner-style test caught the second case crashing with a traceback.

(`_env_file=None` disables the `.env` file for one call, which is how tests keep a developer's real `.env` out of the way.)

## Part 7: Dataclass or Pydantic model?

Both declare fields with type hints. They're for different jobs.

| | `@dataclass` | Pydantic `BaseModel` |
|---|---|---|
| Validates and converts input | No | Yes, every time an instance is created |
| Cost | Plain Python, very fast | Validation work (fast, written in Rust, but not free) |
| JSON in and out | By hand (`asdict`, `json`) | `model_validate_json`, `model_dump_json` |
| Settings from the environment | By hand | `BaseSettings` |
| Best for | Internal values your own code creates | **Boundaries**: files, HTTP requests and responses, configuration, LLM output |

**Rule of thumb:** validate at the edges with Pydantic. Inside, once data is trusted, either works. This course uses Pydantic models throughout Project 02 because the same models will become the FastAPI service's request and response schemas in Session 07.

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| A nullable field with no default, expected to be optional | Missing key → `Field required` | Add a default: `= None` |
| Lax mode on data contracts | `"80"` and `True` are silently accepted as numbers | `Field(strict=True)` on fields where a type mismatch is a bug |
| A validator that forgets `return value` | The field becomes `None` | Always return the (possibly cleaned) value |
| Raising `TypeError` or custom exceptions in validators | Escapes as a crash, not a validation error | Raise `ValueError` |
| Printing `error` to end users | Developer-oriented text, with links | Map `error.errors()` to short `field: message` lines |
| Matching on `msg` text in code | Breaks when Pydantic rewords a message | Match on `type` |
| `extra="forbid"` on third-party data | Breaks when they add a field | `extra="ignore"` for others' data; `"forbid"` for your own inputs |
| Tokens as `str` in settings | Leak through `repr`, logs and tracebacks | `SecretStr`, with `.get_secret_value()` only where needed |
| Catching only `ValidationError` around settings | Unparseable values (`SettingsError`) crash with a traceback | Catch both |

## Practical engineering application

From now on, every boundary in your applications has a Pydantic model: the records you load, the API requests you accept and the responses you return (Session 07), the data external services send you (Session 08), and the structured output you ask an LLM for (Session 09), where "the model returned almost-valid JSON" is a daily event. The habits are the same each time: **declare the contract, decide strictness, collect all errors, map them to messages, and keep secrets in `SecretStr`.**

## Key takeaways

- A Pydantic model validates and converts at run time; one `ValidationError` reports every problem, with `loc`, `type` and `msg`.
- Only a default makes a field optional; `X | None` alone means "required, nullable".
- Use `Field(...)` constraints and `Literal` for rules, and `strict=True` where a type mismatch means a bug.
- `@field_validator` cleans and checks one field (after or before conversion); `@model_validator` checks rules across fields. Raise `ValueError`.
- `model_dump`, `model_dump_json`, `include`/`exclude` and aliases control output; `model_validate_json` reads it back.
- `BaseSettings` loads typed settings from the environment and `.env` (the environment wins); `SecretStr` keeps tokens out of output.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** test yourself with **`02-practice-quiz.ipynb`**, then start the workbook, which begins **Project 02**.